<a href="https://colab.research.google.com/github/JovanT/Fine-tuning-and-Optimizing-Small-Language-Models/blob/test-jovan/notebooks/basic_sft_example.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Basic SFT example

This notebook runs supervised fine-tuning (SFT) against the model `Qwen/Qwen3-0.6B-Base` on the dataset `openai/gsm8k`. The dataset includes 8.5K grade school math questions and answers. The notebook loads the dataset, evaluates and trains the model, and reviews the results. For more information on preparing your environment, see the repository [README](../README.md).

<a href="https://colab.research.google.com/github/ned1313/Fine-tuning-and-Optimizing-Small-Language-Models/blob/main/notebooks/basic_sft_example.ipynb" target="_parent">
  <img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/>
</a>

## Google Colab prep

If you are running this notebook in Google Colab, select a **T4 instance** and run the code block below to install the necessary packages. If you are running locally with the project dependencies already installed, skip this cell.

The install cell installs packages only. Clone or upload the repository into the Colab runtime, including its datasets, and set the kernel's working directory to the repository's `notebooks` directory before continuing. This also keeps relative data, credential, and artifact paths consistent with local runs. Preserve any generated artifacts you need before the runtime is discarded.

For local runs, also set the kernel's working directory to the repository's `notebooks` directory. Opening the notebook in an editor does not by itself guarantee that working directory.

In [1]:
%pip install datasets numpy transformers trl tqdm huggingface_hub

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 996.6/996.6 kB 10.3 MB/s  0:00:00


## Data loading and model selection

This first cell downloads the GSM8K dataset and keeps the train and test splits we will use for fine-tuning and evaluation.

In [2]:
from datasets import load_dataset

MODEL_ID = "Qwen/Qwen3-0.6B-Base"
DS_NAME = "openai/gsm8k"
DS_CONFIG = "main"

full_ds = load_dataset(DS_NAME, DS_CONFIG)

README.md:   0%|          | 0.00/7.93k [00:00<?, ?B/s]

main/train-00000-of-00001.parquet: reconstructing file:   0%|          |  0.00B / 2.31MB            

main/train-00000-of-00001.parquet: downloading bytes:           |  0.00B            

main/test-00000-of-00001.parquet: reconstructing file:   0%|          |  0.00B /  419kB            

main/test-00000-of-00001.parquet: downloading bytes:           |  0.00B            

Generating train split:   0%|          | 0/7473 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/1319 [00:00<?, ? examples/s]

In [3]:
len(full_ds["train"]), len(full_ds["test"])

(7473, 1319)

The next cell sets up the model we want to train, loading both the model and the tokenizer.

In [4]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer

model = AutoModelForCausalLM.from_pretrained(MODEL_ID, device_map="auto", trust_remote_code=True)
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID, trust_remote_code=True)
tokenizer.padding_side = "right"
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

config.json:   0%|          | 0.00/727 [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B / 1.19GB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/310 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/138 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/9.68k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/7.03M [00:00<?, ?B/s]

In [5]:
print(model.config)

print(type(model))

Qwen3Config {
  "architectures": [
    "Qwen3ForCausalLM"
  ],
  "attention_bias": false,
  "attention_dropout": 0.0,
  "bos_token_id": 151643,
  "dtype": "bfloat16",
  "eos_token_id": 151643,
  "head_dim": 128,
  "hidden_act": "silu",
  "hidden_size": 1024,
  "initializer_range": 0.02,
  "intermediate_size": 3072,
  "layer_types": [
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention"
  ],
  "max_position_embeddings": 32768,
  "max_w

## Data formatting

This section turns each GSM8K example into a chat-style prompt/response pair so the model sees a natural instruction-following format during training.

In [6]:
def format_gsm8k_example(example: dict) -> dict:
    question = example["question"].strip()
    answer = example["answer"].strip()
    return {
        "messages": [
            {"role": "user", "content": question},
            {"role": "assistant", "content": answer},
        ],
    }


train_ds = full_ds["train"].map(
    format_gsm8k_example,
    remove_columns=full_ds["train"].column_names,
)

eval_ds = full_ds["test"].map(
    format_gsm8k_example,
    remove_columns=full_ds["test"].column_names,
)

Map:   0%|          | 0/7473 [00:00<?, ? examples/s]

Map:   0%|          | 0/1319 [00:00<?, ? examples/s]

In [7]:
train_ds[0]

{'messages': [{'content': 'Natalia sold clips to 48 of her friends in April, and then she sold half as many clips in May. How many clips did Natalia sell altogether in April and May?',
   'role': 'user'},
  {'content': 'Natalia sold 48/2 = <<48/2=24>>24 clips in May.\nNatalia sold 48+24 = <<48+24=72>>72 clips altogether in April and May.\n#### 72',
   'role': 'assistant'}]}

## Evaluate the baseline

Evaluate the first 100 GSM8K test questions using greedy decoding before training. The scorer compares the last number extracted from the generated answer with the reference final answer. This is a small diagnostic sample with a heuristic scorer, not a full GSM8K benchmark result. Inspect extraction failures and sample answers alongside the accuracy.

In [8]:
import re
from tqdm.auto import tqdm


def build_prompt(tokenizer, question: str) -> str:
    # Match the chat format the model was fine-tuned on.
    messages = [{"role": "user", "content": question}]
    return tokenizer.apply_chat_template(
        messages, tokenize=False, add_generation_prompt=True
    )


def extract_gold_answer(answer: str) -> str:
    # GSM8K gold answers end with a "#### <final answer>" marker.
    marker = answer.rsplit("####", 1)
    text = marker[1] if len(marker) > 1 else answer
    match = re.search(r"-?\d[\d,]*(?:\.\d+)?", text)
    return match.group().replace(",", "") if match else ""


def extract_predicted_answer(response: str) -> str:
    # Use the last number the model produced as its final answer.
    matches = re.findall(r"-?\d[\d,]*(?:\.\d+)?", response)
    return matches[-1].replace(",", "") if matches else ""


def evaluate_model(model, tokenizer, questions, answers, desc="Evaluating", batch_size=8, max_new_tokens=256):
    correct = 0
    total = len(questions)

    was_training = model.training
    prev_use_cache = getattr(model.config, "use_cache", None)
    prev_padding_side = tokenizer.padding_side
    model.eval()
    model.config.use_cache = True
    # Decoder-only models must be left-padded for correct batched generation.
    tokenizer.padding_side = "left"

    try:
        for start in tqdm(range(0, total, batch_size), desc=desc, unit="batch"):
            batch_questions = questions[start:start + batch_size]
            batch_answers = answers[start:start + batch_size]

            prompts = [build_prompt(tokenizer, q) for q in batch_questions]
            inputs = tokenizer(prompts, return_tensors="pt", padding=True).to(model.device)

            with torch.no_grad():
                outputs = model.generate(
                    **inputs,
                    max_new_tokens=max_new_tokens,
                    do_sample=False,
                    pad_token_id=tokenizer.pad_token_id,
                )

            # Only decode the newly generated tokens, not the echoed prompt.
            generated = outputs[:, inputs["input_ids"].shape[1]:]
            responses = tokenizer.batch_decode(generated, skip_special_tokens=True)

            for response, gold in zip(responses, batch_answers):
                if extract_predicted_answer(response) == extract_gold_answer(gold):
                    correct += 1
    finally:
        tokenizer.padding_side = prev_padding_side
        if prev_use_cache is not None:
            model.config.use_cache = prev_use_cache
        if was_training:
            model.train()

    acc = correct / total if total else 0.0
    print(f"{desc}: {correct}/{total} correct ({acc:.2%})")
    return acc

questions = [ex["question"] for ex in full_ds["test"].select(range(100))]
answers = [ex["answer"] for ex in full_ds["test"].select(range(100))]

baseline_acc = evaluate_model(model, tokenizer, questions, answers, desc="Baseline evaluation")

print(f"Baseline accuracy: {baseline_acc:.2%}")

Baseline evaluation:   0%|          | 0/13 [00:00<?, ?batch/s]

Baseline evaluation: 5/100 correct (5.00%)
Baseline accuracy: 5.00%


## Training configuration

This section sets the trainer arguments, including the batch size, learning rate, mixed precision settings, and output directory. You can adjust the `SFTConfig` settings to see how they impact training time and performance.

In [13]:
from pathlib import Path
import random

import numpy as np
from trl import SFTConfig, SFTTrainer

REPO_ROOT = Path.cwd().resolve().parent
ARTIFACTS_DIR = REPO_ROOT / "artifacts"
RUN_NAME = "sft-gsm8k"
RUN_DIR = ARTIFACTS_DIR / RUN_NAME

if torch.cuda.is_available():
    bf16 = torch.cuda.is_bf16_supported()
    fp16 = not bf16
else:
    bf16 = False
    fp16 = False

config = SFTConfig(
    max_length=512,
    per_device_train_batch_size=2,
    per_device_eval_batch_size=1,
    num_train_epochs=1,
    learning_rate=2e-5,
    bf16=bf16,
    fp16=fp16,
    output_dir=str(RUN_DIR),
    gradient_checkpointing=True,
    gradient_accumulation_steps=1,
    lr_scheduler_type="linear",
    eval_strategy="no",
    logging_steps=100,
    completion_only_loss=True,
)

trainer = SFTTrainer(
    args=config,
    model=model,
    processing_class=tokenizer,
    train_dataset=train_ds,
    eval_dataset=eval_ds,
)


def set_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)


set_seed(42)
trainer.train()

trainer.save_model(str(RUN_DIR))

torch.cuda.is_available()=True - bf16=False - fp16 True


RuntimeError: gemm input type at::BFloat16 and output type float is only supported for CUDA devices with compute capability 8.0 or higher

## Reload and evaluate

Release the training model, reload the fine-tuned checkpoint, and evaluate the same first 100 GSM8K test questions with greedy decoding and the same last-number extraction heuristic. Compare the before/after accuracy on this diagnostic sample, and inspect sample answers and extraction failures before drawing broader conclusions.

In [ ]:
import gc

del model
del trainer

gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()      # release cached blocks to CUDA driver
    torch.cuda.ipc_collect()

In [ ]:
# Load model and tokenizer from the fine-tuned checkpoint
model = AutoModelForCausalLM.from_pretrained(str(RUN_DIR), device_map="auto", trust_remote_code=True)
tokenizer = AutoTokenizer.from_pretrained(str(RUN_DIR), trust_remote_code=True)

fine_tuned_acc = evaluate_model(model, tokenizer, questions, answers, desc="Fine-tuned evaluation")

print(f"Baseline accuracy: {baseline_acc:.2%}")
print(f"Fine-tuned accuracy: {fine_tuned_acc:.2%}")
print(f"Improvement: {fine_tuned_acc - baseline_acc:.2%}")

## Upload the trained model (optional)

The following cells publish model, tokenizer, and model-card files to your Hugging Face account under `RUN_NAME`. Verify the destination and repository visibility before running them, and ensure the uploaded files contain no credentials or sensitive data. Skip this section to keep the model local.

If running locally, create a `.env` file in the `notebooks` directory and populate it with the following:

```text
HF_TOKEN=YOUR_TOKEN_VALUE
```

The `.env` file is ignored by git, so your token won't get committed to source control. For directions on generating a token, check the [documentation here](https://huggingface.co/docs/hub/security-tokens) on Hugging Face. You will need to create a token with `write` access.

In Colab, use Colab Secrets and explicitly load the value into the notebook kernel's environment before the client or upload cell.

In [ ]:
# Execute this code block for local runs only
import os

for raw_line in open(".env"):
    line = raw_line.strip()
    key, value = line.split("=", 1)
    os.environ[key] = value


In [ ]:
from huggingface_hub import HfApi

# Push the fine-tuned model to the Hugging Face Hub
model.push_to_hub(RUN_NAME)
tokenizer.push_to_hub(RUN_NAME)

# Push the README.md file to the Hugging Face Hub
api = HfApi()
info = api.whoami()

api.upload_file(
    path_or_fileobj=str(RUN_DIR / "README.md"),
    path_in_repo="README.md",
    repo_id=f"{info['name']}/{RUN_NAME}",
    repo_type="model",
)

## Release used resources

Release the model reference used here and clear unused CUDA cache blocks. Other live references can keep memory allocated; restart the kernel when a full reset is needed.

In [ ]:
del model
gc.collect()

if torch.cuda.is_available():
    torch.cuda.empty_cache()      # release cached blocks to CUDA driver
    torch.cuda.ipc_collect()